# NIFTY 500 explorer
Run `python download.py` first. Logic lives in `lib/`; this notebook only explores and charts.

In [ ]:
import sys; sys.path.insert(0, '..')
%load_ext autoreload
%autoreload 2
import pandas as pd, vectorbt as vbt
from lib import data, strategies, backtest, plotting
d = data.load(start=None)
c = d['Close']
c.shape

## Backtest SMA crossover on all stocks

In [ ]:
entries, exits = strategies.sma_cross(c, 50, 200)
entries.columns = exits.columns = c.columns
pf = backtest.run(c, entries, exits, volume=d['Volume'])
s = backtest.summary(pf)
s.head(20)

In [ ]:
s[['cagr','sharpe','max_dd']].describe()

## Candlestick with trades

In [ ]:
sym = 'RELIANCE.NS'
o = data.ohlcv_for(sym, start='2015-01-01')
fig = plotting.candles(o, sym, entries[sym].loc[o.index], exits[sym].loc[o.index],
    overlays={'SMA50': o.Close.rolling(50).mean(), 'SMA200': o.Close.rolling(200).mean()})
fig.show()

In [ ]:
pf[sym].plot().show()   # vectorbt: orders, trade PnL, equity

## Parameter sweep (one stock)

In [ ]:
fast = vbt.MA.run(c[sym], [10,20,50], short_name='f')
slow = vbt.MA.run(c[sym], [100,150,200], short_name='s')
ent = fast.ma_crossed_above(slow); ex = fast.ma_crossed_below(slow)
pfs = vbt.Portfolio.from_signals(c[sym], ent, ex, fees=0.0011, freq='1D')
pfs.sharpe_ratio().vbt.heatmap(x_level='f_window', y_level='s_window', symmetric=False)